# 03. Feature Engineering Data Preparation

이 노트북은 파생변수 생성에 사용하기 전 외부 입지 데이터를 정제한다.

## 정제 대상

- 병원정보서비스 2025년 3월, 6월, 9월, 12월 파일에서 서울 소재 상급종합/종합병원만 비교
- 서울시 역사마스터 정보는 별도 필터링 없이 원본 그대로 사용
- 서울시 대규모점포 인허가 정보에서 영업 중인 대형마트만 필터링

원본 파일은 `data/external/`에 유지하고, 정제 결과는 `data/interim/`에 저장한다.

## 1. 라이브러리 및 경로 설정

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any
import os
import re
import time
import unicodedata

import numpy as np
import pandas as pd
import requests
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
EXTERNAL_DIR = PROJECT_ROOT / 'data/external'
INTERIM_DIR = PROJECT_ROOT / 'data/interim'
PROCESSED_DIR = PROJECT_ROOT / 'data/processed'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

EXTERNAL_DIR


### 1-1. 외부 데이터 정제 함수 정의

제출 노트북을 단독 실행할 수 있도록 외부 데이터 정제 및 거리 계산 함수를 이 노트북 안에 정의한다.


In [ ]:
HOSPITAL_TARGET_TYPES = ("상급종합", "종합병원")

CBD_CENTERS = {
    "CBD": {
        "name": "도심권",
        "description": "시청/광화문 일대",
        "address": "서울특별시 중구 세종대로 110",
        "latitude": 37.5665,
        "longitude": 126.9780,
    },
    "YBD": {
        "name": "여의도권",
        "description": "여의도 일대",
        "address": "서울특별시 영등포구 의사당대로 1",
        "latitude": 37.5259,
        "longitude": 126.9209,
    },
    "GBD": {
        "name": "강남권",
        "description": "강남역 사거리 일대",
        "address": "서울특별시 강남구 강남대로 396",
        "latitude": 37.4979,
        "longitude": 127.0276,
    },
}


def normalize_filename(name: str) -> str:
    return unicodedata.normalize("NFC", name)


def find_external_file(external_dir: Path, keyword: str) -> Path:
    matches = [
        path
        for path in external_dir.glob("*.csv")
        if keyword in normalize_filename(path.name)
    ]
    if not matches:
        raise FileNotFoundError(f"{external_dir}에서 '{keyword}' CSV 파일을 찾지 못했습니다.")
    if len(matches) > 1:
        raise ValueError(f"'{keyword}' 파일이 여러 개입니다: {[p.name for p in matches]}")
    return matches[0]


def get_hospital_quarter(path: Path) -> str:
    match = re.search(r"2025\.(\d+)", normalize_filename(path.name))
    if not match:
        raise ValueError(f"파일명에서 분기 월을 찾지 못했습니다: {path.name}")
    return match.group(1).zfill(2)


def load_target_hospitals(path: Path) -> pd.DataFrame:
    columns = [
        "암호화요양기호",
        "요양기관명",
        "종별코드명",
        "시도코드명",
        "시군구코드명",
        "주소",
        "전화번호",
        "개설일자",
        "총의사수",
        "좌표(X)",
        "좌표(Y)",
    ]
    df = pd.read_csv(path, encoding="utf-8-sig", usecols=columns)
    df = df[
        df["시도코드명"].eq("서울")
        & df["종별코드명"].isin(HOSPITAL_TARGET_TYPES)
    ].copy()
    df["snapshot_month"] = get_hospital_quarter(path)
    df = df.rename(
        columns={
            "암호화요양기호": "hospital_id",
            "요양기관명": "hospital_name",
            "종별코드명": "hospital_type",
            "시도코드명": "sido",
            "시군구코드명": "gu",
            "주소": "address",
            "전화번호": "phone",
            "개설일자": "opened_date",
            "총의사수": "doctor_count",
            "좌표(X)": "longitude",
            "좌표(Y)": "latitude",
        }
    )
    return df[
        [
            "snapshot_month",
            "hospital_id",
            "hospital_name",
            "hospital_type",
            "sido",
            "gu",
            "address",
            "phone",
            "opened_date",
            "doctor_count",
            "longitude",
            "latitude",
        ]
    ].sort_values(["hospital_type", "gu", "hospital_name"]).reset_index(drop=True)


def compare_hospital_snapshots(hospital_by_month: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows = []
    diff_rows = []
    months = sorted(hospital_by_month)

    for month, df in hospital_by_month.items():
        counts = df["hospital_type"].value_counts()
        summary_rows.append(
            {
                "snapshot_month": month,
                "total_count": len(df),
                "tertiary_count": int(counts.get("상급종합", 0)),
                "general_count": int(counts.get("종합병원", 0)),
            }
        )

    base_month = months[0]
    base_ids = set(hospital_by_month[base_month]["hospital_id"])
    for month in months[1:]:
        current = hospital_by_month[month]
        current_ids = set(current["hospital_id"])
        added = current[current["hospital_id"].isin(current_ids - base_ids)]
        removed = hospital_by_month[base_month][
            hospital_by_month[base_month]["hospital_id"].isin(base_ids - current_ids)
        ]

        for _, row in added.iterrows():
            diff_rows.append(
                {
                    "base_month": base_month,
                    "compare_month": month,
                    "change_type": "added",
                    "hospital_name": row["hospital_name"],
                    "hospital_type": row["hospital_type"],
                    "gu": row["gu"],
                    "address": row["address"],
                }
            )
        for _, row in removed.iterrows():
            diff_rows.append(
                {
                    "base_month": base_month,
                    "compare_month": month,
                    "change_type": "removed",
                    "hospital_name": row["hospital_name"],
                    "hospital_type": row["hospital_type"],
                    "gu": row["gu"],
                    "address": row["address"],
                }
            )

    summary = pd.DataFrame(summary_rows).sort_values("snapshot_month")
    diffs = pd.DataFrame(diff_rows)
    return summary, diffs


def clean_large_marts(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="cp949")
    df = df[
        df["업태구분명"].eq("대형마트")
        & df["영업상태명"].eq("영업/정상")
        & df["상세영업상태명"].eq("정상영업")
    ].copy()

    df["address"] = df["도로명주소"].fillna(df["지번주소"])
    df["gu"] = df["address"].str.extract(r"서울특별시\s+([^\s]+)")[0]
    for column in ["좌표정보(X)", "좌표정보(Y)"]:
        df[column] = pd.to_numeric(
            df[column].astype(str).str.strip().replace("", pd.NA),
            errors="coerce",
        )

    df = df.rename(
        columns={
            "관리번호": "store_id",
            "사업장명": "store_name",
            "업태구분명": "store_type",
            "영업상태명": "business_status",
            "상세영업상태명": "business_status_detail",
            "인허가일자": "licensed_date",
            "좌표정보(X)": "coord_x",
            "좌표정보(Y)": "coord_y",
        }
    )
    return df[
        [
            "store_id",
            "store_name",
            "store_type",
            "business_status",
            "business_status_detail",
            "licensed_date",
            "gu",
            "address",
            "coord_x",
            "coord_y",
        ]
    ].sort_values(["gu", "store_name"]).reset_index(drop=True)


def require_coordinate_columns(
    df: pd.DataFrame,
    latitude_col: str = "latitude",
    longitude_col: str = "longitude",
    label: str = "dataframe",
) -> None:
    missing = [col for col in [latitude_col, longitude_col] if col not in df.columns]
    if missing:
        raise KeyError(f"{label}에 좌표 컬럼이 없습니다: {missing}")


def haversine_distance_km(
    left_latitude,
    left_longitude,
    right_latitude,
    right_longitude,
) -> np.ndarray:
    radius_km = 6371.0088
    left_latitude = np.radians(left_latitude)
    left_longitude = np.radians(left_longitude)
    right_latitude = np.radians(right_latitude)
    right_longitude = np.radians(right_longitude)

    delta_latitude = right_latitude - left_latitude
    delta_longitude = right_longitude - left_longitude
    a = (
        np.sin(delta_latitude / 2) ** 2
        + np.cos(left_latitude)
        * np.cos(right_latitude)
        * np.sin(delta_longitude / 2) ** 2
    )
    return radius_km * 2 * np.arcsin(np.sqrt(a))


def nearest_facility_distance_km(
    locations: pd.DataFrame,
    facilities: pd.DataFrame,
    location_latitude_col: str = "latitude",
    location_longitude_col: str = "longitude",
    facility_latitude_col: str = "latitude",
    facility_longitude_col: str = "longitude",
    chunk_size: int = 5000,
) -> pd.Series:
    require_coordinate_columns(
        locations, location_latitude_col, location_longitude_col, "locations"
    )
    require_coordinate_columns(
        facilities, facility_latitude_col, facility_longitude_col, "facilities"
    )
    facilities = facilities.dropna(
        subset=[facility_latitude_col, facility_longitude_col]
    )
    if facilities.empty:
        return pd.Series(np.nan, index=locations.index)

    facility_latitudes = facilities[facility_latitude_col].to_numpy(dtype=float)
    facility_longitudes = facilities[facility_longitude_col].to_numpy(dtype=float)
    nearest = pd.Series(np.nan, index=locations.index, dtype=float)
    valid_locations = locations.dropna(
        subset=[location_latitude_col, location_longitude_col]
    )

    for start in range(0, len(valid_locations), chunk_size):
        end = min(start + chunk_size, len(valid_locations))
        chunk = valid_locations.iloc[start:end]
        distances = haversine_distance_km(
            chunk[location_latitude_col].to_numpy(dtype=float)[:, None],
            chunk[location_longitude_col].to_numpy(dtype=float)[:, None],
            facility_latitudes[None, :],
            facility_longitudes[None, :],
        )
        nearest.loc[chunk.index] = np.nanmin(distances, axis=1)

    return nearest


def nearby_facility_count(
    locations: pd.DataFrame,
    facilities: pd.DataFrame,
    radius_km: float = 1.0,
    location_latitude_col: str = "latitude",
    location_longitude_col: str = "longitude",
    facility_latitude_col: str = "latitude",
    facility_longitude_col: str = "longitude",
    chunk_size: int = 5000,
) -> pd.Series:
    require_coordinate_columns(
        locations, location_latitude_col, location_longitude_col, "locations"
    )
    require_coordinate_columns(
        facilities, facility_latitude_col, facility_longitude_col, "facilities"
    )
    facilities = facilities.dropna(
        subset=[facility_latitude_col, facility_longitude_col]
    )
    if facilities.empty:
        return pd.Series(pd.NA, index=locations.index, dtype="Int64")

    facility_latitudes = facilities[facility_latitude_col].to_numpy(dtype=float)
    facility_longitudes = facilities[facility_longitude_col].to_numpy(dtype=float)
    counts = pd.Series(pd.NA, index=locations.index, dtype="Int64")
    valid_locations = locations.dropna(
        subset=[location_latitude_col, location_longitude_col]
    )

    for start in range(0, len(valid_locations), chunk_size):
        end = min(start + chunk_size, len(valid_locations))
        chunk = valid_locations.iloc[start:end]
        distances = haversine_distance_km(
            chunk[location_latitude_col].to_numpy(dtype=float)[:, None],
            chunk[location_longitude_col].to_numpy(dtype=float)[:, None],
            facility_latitudes[None, :],
            facility_longitudes[None, :],
        )
        counts.loc[chunk.index] = (distances <= radius_km).sum(axis=1)

    return counts


def add_cbd_distance_features(
    apartments: pd.DataFrame,
    latitude_col: str = "latitude",
    longitude_col: str = "longitude",
) -> pd.DataFrame:
    require_coordinate_columns(apartments, latitude_col, longitude_col, "apartments")
    result = apartments.copy()
    distance_columns = []

    for code, center in CBD_CENTERS.items():
        column = f"distance_to_{code.lower()}_km"
        result[column] = haversine_distance_km(
            result[latitude_col].to_numpy(dtype=float),
            result[longitude_col].to_numpy(dtype=float),
            center["latitude"],
            center["longitude"],
        )
        distance_columns.append(column)

    distances = result[distance_columns]
    result["nearest_business_district_distance_km"] = distances.min(axis=1)
    result["nearest_business_district"] = pd.NA
    valid_distances = distances.notna().any(axis=1)
    result.loc[valid_distances, "nearest_business_district"] = (
        distances.loc[valid_distances]
        .idxmin(axis=1)
        .str.replace("distance_to_", "", regex=False)
        .str.replace("_km", "", regex=False)
        .str.upper()
    )
    return result


def add_accessibility_features(
    apartments: pd.DataFrame,
    stations: pd.DataFrame,
    hospitals: pd.DataFrame,
    large_marts: pd.DataFrame | None = None,
    latitude_col: str = "latitude",
    longitude_col: str = "longitude",
) -> pd.DataFrame:
    result = add_cbd_distance_features(apartments, latitude_col, longitude_col)

    result["nearest_subway_distance_km"] = nearest_facility_distance_km(
        result,
        stations,
        location_latitude_col=latitude_col,
        location_longitude_col=longitude_col,
        facility_latitude_col="latitude",
        facility_longitude_col="longitude",
    )
    result["hospital_count_within_1km"] = nearby_facility_count(
        result,
        hospitals,
        radius_km=1.0,
        location_latitude_col=latitude_col,
        location_longitude_col=longitude_col,
        facility_latitude_col="latitude",
        facility_longitude_col="longitude",
    )
    result["nearest_hospital_distance_km"] = nearest_facility_distance_km(
        result,
        hospitals,
        location_latitude_col=latitude_col,
        location_longitude_col=longitude_col,
        facility_latitude_col="latitude",
        facility_longitude_col="longitude",
    )

    if large_marts is not None:
        result["large_mart_count_within_1km"] = nearby_facility_count(
            result,
            large_marts,
            radius_km=1.0,
            location_latitude_col=latitude_col,
            location_longitude_col=longitude_col,
            facility_latitude_col="latitude",
            facility_longitude_col="longitude",
        )

    return result


### 1-2. Naver 지오코딩 함수 정의

제출 노트북을 단독 실행할 수 있도록 지오코딩, 재시도, 캐시 병합 함수를 이 노트북 안에 정의한다.


In [ ]:
NAVER_GEOCODE_URL = "https://maps.apigw.ntruss.com/map-geocode/v2/geocode"
RETRYABLE_CACHE_STATUSES = ("error",)
GEOCODE_RESULT_COLUMNS = [
    "geocode_status",
    "matched_address",
    "road_address",
    "jibun_address",
    "latitude",
    "longitude",
    "error_message",
]


def load_env_file(path: Path = PROJECT_ROOT / ".env") -> None:
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip('"').strip("'")


def get_naver_credentials() -> tuple[str, str]:
    load_env_file()
    client_id = (
        os.getenv("NAVER_CLIENT_ID")
        or os.getenv("NAVER_MAPS_CLIENT_ID")
        or os.getenv("NAVER_API_KEY_ID")
    )
    client_secret = (
        os.getenv("NAVER_CLIENT_SECRET")
        or os.getenv("NAVER_MAPS_CLIENT_SECRET")
        or os.getenv("NAVER_API_KEY")
    )
    if not client_id or not client_secret:
        raise RuntimeError(
            ".env에 NAVER_CLIENT_ID와 NAVER_CLIENT_SECRET을 설정해야 합니다."
        )
    return client_id, client_secret


def mask_secret(value: str) -> str:
    if len(value) <= 8:
        return "*" * len(value)
    return f"{value[:4]}...{value[-4:]}"


def is_too_coarse_address(address: str) -> bool:
    return bool(re.fullmatch(r"서울특별시\s+\S+구", str(address).strip()))


def normalize_address_value(value: Any) -> str | None:
    if pd.isna(value):
        return None
    address = str(value).strip()
    if not address or address.lower() == "nan":
        return None
    return address


def geocode_missing_address_row(address_col: str = "full_road_address") -> dict[str, Any]:
    return {
        address_col: pd.NA,
        "geocode_status": "missing_address",
        "matched_address": pd.NA,
        "road_address": pd.NA,
        "jibun_address": pd.NA,
        "latitude": pd.NA,
        "longitude": pd.NA,
        "error_message": "address is missing or blank",
    }


def test_naver_geocoding(address: str = "서울특별시 중구 세종대로 110") -> dict[str, Any]:
    client_id, client_secret = get_naver_credentials()
    headers = {
        "X-NCP-APIGW-API-KEY-ID": client_id,
        "X-NCP-APIGW-API-KEY": client_secret,
    }
    response = requests.get(
        NAVER_GEOCODE_URL,
        headers=headers,
        params={"query": address},
        timeout=10,
    )
    try:
        body = response.json()
    except ValueError:
        body = response.text[:500]
    return {
        "client_id": mask_secret(client_id),
        "client_secret": mask_secret(client_secret),
        "status_code": response.status_code,
        "body": body,
    }


def validate_naver_geocoding() -> None:
    result = test_naver_geocoding()
    if result["status_code"] != 200:
        raise RuntimeError(f"Naver Geocoding 인증/연결 테스트 실패: {result}")


def geocode_address(
    address: str,
    client_id: str,
    client_secret: str,
    address_col: str = "full_road_address",
    timeout: int = 10,
) -> dict[str, Any]:
    normalized_address = normalize_address_value(address)
    if normalized_address is None:
        return geocode_missing_address_row(address_col=address_col)

    address = normalized_address
    if is_too_coarse_address(address):
        return {
            address_col: address,
            "geocode_status": "too_coarse",
            "matched_address": pd.NA,
            "road_address": pd.NA,
            "jibun_address": pd.NA,
            "latitude": pd.NA,
            "longitude": pd.NA,
        }

    headers = {
        "X-NCP-APIGW-API-KEY-ID": client_id,
        "X-NCP-APIGW-API-KEY": client_secret,
    }
    response = requests.get(
        NAVER_GEOCODE_URL,
        headers=headers,
        params={"query": address},
        timeout=timeout,
    )
    response.raise_for_status()
    payload = response.json()
    addresses = payload.get("addresses", [])

    if not addresses:
        return {
            address_col: address,
            "geocode_status": "not_found",
            "matched_address": pd.NA,
            "road_address": pd.NA,
            "jibun_address": pd.NA,
            "latitude": pd.NA,
            "longitude": pd.NA,
        }

    match = addresses[0]
    return {
        address_col: address,
        "geocode_status": "ok",
        "matched_address": match.get("roadAddress") or match.get("jibunAddress"),
        "road_address": match.get("roadAddress"),
        "jibun_address": match.get("jibunAddress"),
        "latitude": float(match["y"]),
        "longitude": float(match["x"]),
    }


def geocode_error_row(
    address: str,
    error_message: str,
    address_col: str = "full_road_address",
) -> dict[str, Any]:
    return {
        address_col: address,
        "geocode_status": "error",
        "matched_address": pd.NA,
        "road_address": pd.NA,
        "jibun_address": pd.NA,
        "latitude": pd.NA,
        "longitude": pd.NA,
        "error_message": error_message,
    }


def retry_delay_seconds(exc: Exception, attempt: int, base_delay: float) -> float | None:
    if isinstance(exc, requests.HTTPError):
        status_code = exc.response.status_code if exc.response is not None else None
        if status_code == 429:
            retry_after = exc.response.headers.get("Retry-After") if exc.response else None
            if retry_after:
                try:
                    return float(retry_after)
                except ValueError:
                    pass
            return base_delay * (2 ** (attempt - 1))
        if status_code is not None and 500 <= status_code < 600:
            return base_delay * (2 ** (attempt - 1))
        return None

    if isinstance(
        exc,
        (
            requests.ConnectionError,
            requests.Timeout,
            requests.RequestException,
            ValueError,
        ),
    ):
        return base_delay * (2 ** (attempt - 1))

    return None


def geocode_address_with_retry(
    address: str,
    client_id: str,
    client_secret: str,
    address_col: str = "full_road_address",
    timeout: int = 10,
    max_retries: int = 3,
    retry_base_delay: float = 1.0,
) -> dict[str, Any]:
    last_error: Exception | None = None
    for attempt in range(1, max_retries + 2):
        try:
            return geocode_address(
                address,
                client_id,
                client_secret,
                address_col=address_col,
                timeout=timeout,
            )
        except Exception as exc:
            last_error = exc
            delay = retry_delay_seconds(exc, attempt, retry_base_delay)
            if delay is None or attempt > max_retries:
                break
            time.sleep(delay)

    return geocode_error_row(
        address,
        str(last_error) if last_error is not None else "unknown geocoding error",
        address_col=address_col,
    )


def load_existing_cache(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, encoding="utf-8-sig")


def prepare_geocode_result(
    rows: list[dict[str, Any]],
    address_col: str,
) -> pd.DataFrame:
    result = pd.DataFrame(rows)
    for column in [address_col, *GEOCODE_RESULT_COLUMNS]:
        if column not in result.columns:
            result[column] = pd.NA
    result = result.drop_duplicates(subset=[address_col], keep="last")
    result = result.sort_values(address_col).reset_index(drop=True)
    return result


def save_geocode_rows(
    rows: list[dict[str, Any]],
    output_path: Path,
    address_col: str,
) -> pd.DataFrame:
    result = prepare_geocode_result(rows, address_col)
    result.to_csv(output_path, index=False, encoding="utf-8-sig")
    return result


def geocode_unique_addresses(
    input_path: Path,
    output_path: Path,
    address_col: str = "full_road_address",
    sleep_seconds: float = 0.1,
    limit: int | None = None,
    force: bool = False,
    max_workers: int = 5,
    max_retries: int = 3,
    retry_base_delay: float = 1.0,
    retry_cache_statuses: tuple[str, ...] = RETRYABLE_CACHE_STATUSES,
) -> pd.DataFrame:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    source = pd.read_csv(input_path, encoding="utf-8-sig", usecols=[address_col])
    normalized_addresses = source[address_col].map(normalize_address_value)
    missing_address_count = normalized_addresses.isna().sum()
    addresses = (
        normalized_addresses.dropna()
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    if limit is not None:
        addresses = addresses[:limit]

    cache = load_existing_cache(output_path)
    cached_addresses = set()
    if not force and not cache.empty and address_col in cache.columns:
        cache[address_col] = cache[address_col].map(normalize_address_value)
        if "geocode_status" in cache.columns:
            reusable_cache = cache[
                ~cache["geocode_status"].fillna("").isin(retry_cache_statuses)
            ]
        else:
            reusable_cache = cache
        cached_addresses = set(reusable_cache[address_col].dropna())

    rows = [] if force or cache.empty else cache.to_dict("records")
    pending = [address for address in addresses if address not in cached_addresses]

    if missing_address_count:
        print(f"missing or blank addresses skipped: {missing_address_count:,} rows")
    if pending:
        print(
            f"geocoding pending addresses: {len(pending):,} "
            f"(max_workers={max_workers}, max_retries={max_retries}, "
            f"retry_cache_statuses={retry_cache_statuses})"
        )
        client_id, client_secret = get_naver_credentials()
        validate_naver_geocoding()

    completed = 0
    max_workers = max(1, max_workers)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_address = {}
        for address in pending:
            future = executor.submit(
                geocode_address_with_retry,
                address,
                client_id,
                client_secret,
                address_col,
                10,
                max_retries,
                retry_base_delay,
            )
            future_to_address[future] = address
            if sleep_seconds > 0:
                time.sleep(sleep_seconds)

        for future in as_completed(future_to_address):
            address = future_to_address[future]
            try:
                rows.append(future.result())
            except Exception as exc:
                rows.append(geocode_error_row(address, str(exc), address_col=address_col))

            completed += 1
            if completed % 100 == 0 or completed == len(pending):
                print(f"geocoded {completed:,}/{len(pending):,} pending addresses")
                save_geocode_rows(rows, output_path, address_col)

        if not pending:
            save_geocode_rows(rows, output_path, address_col)

    result = save_geocode_rows(rows, output_path, address_col)
    if "geocode_status" in result.columns:
        print("geocode status summary:")
        print(result["geocode_status"].value_counts(dropna=False).to_string())
    return result


def merge_coordinates(
    apartment_path: Path,
    geocoded_path: Path,
    output_path: Path,
    address_col: str = "full_road_address",
) -> pd.DataFrame:
    apartments = pd.read_csv(apartment_path, encoding="utf-8-sig")
    coordinates = pd.read_csv(geocoded_path, encoding="utf-8-sig")
    apartments["_geocode_key"] = apartments[address_col].map(normalize_address_value)
    coordinates["_geocode_key"] = coordinates[address_col].map(normalize_address_value)

    coordinate_cols = [
        column for column in GEOCODE_RESULT_COLUMNS if column in coordinates.columns
    ]
    merged = apartments.merge(
        coordinates[["_geocode_key", *coordinate_cols]],
        on="_geocode_key",
        how="left",
    )
    for column in GEOCODE_RESULT_COLUMNS:
        if column not in merged.columns:
            merged[column] = pd.NA

    missing_address_mask = merged["_geocode_key"].isna()
    unmatched_address_mask = (
        merged["_geocode_key"].notna() & merged["geocode_status"].isna()
    )
    merged.loc[missing_address_mask, "geocode_status"] = "missing_address"
    merged.loc[unmatched_address_mask, "geocode_status"] = "not_geocoded"
    if "error_message" not in merged.columns:
        merged["error_message"] = pd.NA
    merged["error_message"] = merged["error_message"].astype("object")
    merged.loc[missing_address_mask, "error_message"] = "address is missing or blank"

    merged = merged.drop(columns=["_geocode_key"])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(output_path, index=False, encoding="utf-8-sig")
    return merged


## 2. 병원정보서비스 분기별 비교

상급종합병원과 종합병원은 개수가 많지 않고 시점별 변화가 가격 접근성 변수에 영향을 줄 수 있으므로, 3월·6월·9월·12월 스냅샷을 먼저 비교한다.

In [2]:
hospital_files = sorted(
    [path for path in EXTERNAL_DIR.glob('*.csv') if '병원정보서비스' in normalize_filename(path.name)],
    key=get_hospital_quarter,
)

hospital_by_month = {
    get_hospital_quarter(path): load_target_hospitals(path)
    for path in hospital_files
}

hospital_summary, hospital_diffs = compare_hospital_snapshots(hospital_by_month)
hospital_summary

,snapshot_month,total_count,tertiary_count,general_count
0,03,58,14,44
1,06,59,14,45
2,09,59,14,45
3,12,59,14,45


In [3]:
hospital_diffs

,base_month,compare_month,change_type,hospital_name,hospital_type,gu,address
0,03,06,added,서울현대병원,종합병원,강북구,"서울특별시 강북구 도봉로 374, (번동, 서울현대병원)"
1,03,09,added,서울현대병원,종합병원,강북구,"서울특별시 강북구 도봉로 374, (번동, 서울현대병원)"
2,03,12,added,서울현대병원,종합병원,강북구,"서울특별시 강북구 도봉로 374, (번동, 서울현대병원)"


비교 결과 2025년 3월은 58개, 6월·9월·12월은 59개이다. 6월 이후 스냅샷은 서로 동일하며, 3월 대비 `서울현대병원` 1개 종합병원이 추가되어 있다.

따라서 연간 분석용 병원 접근성 변수에는 가장 최근 스냅샷인 12월 자료를 대표 병원 목록으로 사용한다.

In [4]:
selected_hospital_month = '12'
seoul_hospitals = hospital_by_month[selected_hospital_month].copy()
seoul_hospitals.head()

,snapshot_month,hospital_id,hospital_name,hospital_type,sido,gu,address,phone,opened_date,doctor_count,longitude,latitude
0,12,JDQ4MTg4MSM1MSMkMSMkMCMkODkkMzgxMzUxIzExIyQxIy...,삼성서울병원,상급종합,서울,강남구,"서울특별시 강남구 일원로 81, (일원동, 삼성의료원)",02-3410-2114,1994.6.13,1285,127.085151,37.488298
1,12,JDQ4MTg4MSM1MSMkMSMkMCMkODkkMzgxMzUxIzExIyQxIy...,연세대학교의과대학 강남세브란스병원,상급종합,서울,강남구,"서울특별시 강남구 언주로 211, 강남세브란스병원 (도곡동)",02-2019-3114,1983.4.4,504,127.046268,37.492930
2,12,JDQ4MTg4MSM1MSMkMSMkMCMkODkkMzgxMzUxIzExIyQxIy...,건국대학교병원,상급종합,서울,광진구,"서울특별시 광진구 능동로 120-1, (화양동)",1588-1533,1982.11.16,357,127.071828,37.540376
3,12,JDQ4MTg4MSM1MSMkMSMkMCMkODkkMzgxMzUxIzExIyQxIy...,고려대학교의과대학부속구로병원,상급종합,서울,구로구,"서울특별시 구로구 구로동로 148, 고려대부속구로병원 (구로동)",02-2626-1114,1983.8.31,497,126.884870,37.492052
4,12,JDQ4MTg4MSM1MSMkMSMkMCMkODkkMzgxMzUxIzExIyQxIy...,경희대학교병원,상급종합,서울,동대문구,"서울특별시 동대문구 경희대로 23, (회기동)",02-958-8114,1971.10.5,394,127.051852,37.594119


## 3. 역사마스터 원본 확인

역사마스터 데이터는 서울시 제공 파일을 그대로 사용한다. 별도 필터링 없이 역사 ID, 역사명, 호선, 위도, 경도 컬럼을 확인한다.

추후 거리 파생변수 계산 단계에서 필요하면 역명·호선 중복이나 환승역 처리 기준만 별도로 정한다.

In [5]:
station_path = find_external_file(EXTERNAL_DIR, '역사마스터')
station_master = pd.read_csv(station_path, encoding='cp949')

station_master.shape, station_master.head()

((783, 5),
    역사_ID 역사명          호선        위도         경도
 0   9010  동탄  수도권 광역급행철도  37.20034  127.09569
 1   9009  구성  수도권 광역급행철도  37.29913  127.10389
 2   9008  성남  수도권 광역급행철도  37.39467  127.12058
 3   9007  수서  수도권 광역급행철도  37.48637  127.10161
 4   9006  삼성  수도권 광역급행철도  37.50887  127.06324)

In [6]:
station_master['호선'].value_counts().head(15)

호선
5호선      56
7호선      53
2호선      50
6호선      39
경부선      39
분당선      34
3호선      34
인천1호선    33
경원선      32
인천2호선    27
경의중앙선    27
4호선      26
9호선      25
중앙선      21
경인선      20
Name: count, dtype: int64

## 4. 영업 중인 대형마트 필터링

대규모점포 인허가 정보에서 `업태구분명 == 대형마트`, `영업상태명 == 영업/정상`, `상세영업상태명 == 정상영업`인 행만 사용한다.

In [7]:
large_store_path = find_external_file(EXTERNAL_DIR, '대규모점포')
large_marts = clean_large_marts(large_store_path)

large_marts.shape, large_marts.head()

((64, 10),
        store_id   store_name store_type business_status  \
 0  2.006320e+18   (주)이마트 역삼점       대형마트           영업/정상   
 1  2.005320e+18    (주)이마트천호점       대형마트           영업/정상   
 2  2.009320e+18     홈플러스 강동점       대형마트           영업/정상   
 3  2.012310e+18  (주)농협유통 미아점       대형마트           영업/정상   
 4  2.005320e+18  홈플러스(주) 강서점       대형마트           영업/정상   
 
   business_status_detail licensed_date   gu                    address  \
 0                   정상영업    2006.10.24  강남구    서울특별시 강남구 역삼로 310 (역삼동)   
 1                   정상영업     2005.3.31  강동구  서울특별시 강동구 천호대로 1017 (천호동)   
 2                   정상영업     2008.11.7  강동구  서울특별시 강동구 양재대로 1571 (천호동)   
 3                   정상영업     2012.3.22  강북구  서울특별시 강북구 도봉로33길 18 (미아동)   
 4                   정상영업     2005.8.26  강서구  서울특별시 강서구  화곡로  398 (등촌동)   
 
        coord_x      coord_y  
 0  204213.6432  444113.0282  
 1  211025.9250  448495.1892  
 2  212501.3691  449282.3104  
 3  202266.4487  457691.8024  
 4  187119.9482  450

In [8]:
large_marts['gu'].value_counts(dropna=False).sort_index()

gu
강남구     1
강동구     2
강북구     1
강서구     1
관악구     1
광진구     6
구로구     3
금천구     3
노원구     2
도봉구     1
동대문구    1
마포구     5
망우로     1
서초구     4
성동구     1
성북구     7
송파구     4
양천구     1
영등포구    6
은평구     4
종로구     1
중구      2
중랑구     5
NaN     1
Name: count, dtype: int64

대형마트 좌표는 원본의 `좌표정보(X)`, `좌표정보(Y)`를 보존한다. 이 좌표는 지하철·병원 위경도와 좌표계가 다르므로, 거리 파생변수 계산 전에는 좌표계 변환 또는 주소 기반 지오코딩이 필요하다.

## 5. Naver 지오코딩으로 좌표 생성 및 상태 점검

아파트 거래 데이터의 `full_road_address` 고유값만 Naver Geocoding API에 요청해 좌표를 생성한다. 전체 거래 행은 77,359건이지만 고유 도로명 주소는 5,769개이므로, 중복 주소를 제거한 뒤 좌표를 구하고 다시 원본 거래 데이터에 병합한다.

대형마트도 반경 1km 카운트를 계산하려면 위경도가 필요하므로, 정제한 대형마트 주소도 같은 방식으로 좌표를 생성한다.

기존 지오코딩 캐시가 있으면 성공 또는 비일시적 실패(`ok`, `not_found`, `too_coarse`)는 재사용하고, 일시적 오류 가능성이 있는 `error` 상태만 다시 요청한다. 실행 후 `geocode_status` 분포를 확인해 좌표화 성공률과 실패 유형을 점검한다.

`.env`에는 다음 값을 설정한다.

```text
NAVER_CLIENT_ID=...
NAVER_CLIENT_SECRET=...
```


In [9]:
apt_path = INTERIM_DIR / 'seoul_apt_trade_2025_basic_cleaned.csv'
apt_geocoded_path = INTERIM_DIR / 'seoul_apt_address_geocoded.csv'
apt_with_coordinates_path = PROCESSED_DIR / 'seoul_apt_trade_2025_with_coordinates.csv'
large_mart_address_path = INTERIM_DIR / 'seoul_large_mart_addresses.csv'
large_mart_geocoded_path = INTERIM_DIR / 'seoul_large_mart_addresses_geocoded.csv'
large_mart_with_coordinates_path = INTERIM_DIR / 'seoul_large_marts_geocoded.csv'
features_output_path = PROCESSED_DIR / 'seoul_apt_trade_2025_features.csv'
features_modeling_ready_path = PROCESSED_DIR / 'seoul_apt_trade_2025_features_modeling_ready.csv'
geocode_unresolved_path = INTERIM_DIR / 'seoul_apt_geocode_unresolved.csv'

apt_df = pd.read_csv(apt_path, encoding='utf-8-sig')
unique_address_count = apt_df['full_road_address'].nunique(dropna=True)
print(f'전체 거래 행: {len(apt_df):,}')
print(f'고유 도로명 주소: {unique_address_count:,}')

전체 거래 행: 77,359
고유 도로명 주소: 5,769


In [10]:
# 키 값을 노출하지 않고 Naver Geocoding 인증 상태만 확인한다.
# status_code가 200이면 인증 성공이다. 실패해도 노트북 실행은 멈추지 않는다.
try:
    auth_test_result = test_naver_geocoding()
    print(auth_test_result)
except Exception as exc:
    auth_test_result = {'status_code': 'failed', 'error': str(exc)}
    print('Naver 지오코딩 테스트 실패:', exc)
    print('인터넷 연결, API 키, 또는 Naver Maps 서비스 상태를 확인하세요.')


{'client_id': 'fw03...zzm2', 'client_secret': 'cjB3...EsWS', 'status_code': 200, 'body': {'status': 'OK', 'meta': {'totalCount': 1, 'page': 1, 'count': 1}, 'addresses': [{'roadAddress': '서울특별시 중구 세종대로 110 서울특별시청', 'jibunAddress': '서울특별시 중구 태평로1가 31 서울특별시청', 'englishAddress': '110, Sejong-daero, Jung-gu, Seoul, Republic of Korea', 'addressElements': [{'types': ['SIDO'], 'longName': '서울특별시', 'shortName': '서울특별시', 'code': ''}, {'types': ['SIGUGUN'], 'longName': '중구', 'shortName': '중구', 'code': ''}, {'types': ['DONGMYUN'], 'longName': '태평로1가', 'shortName': '태평로1가', 'code': ''}, {'types': ['RI'], 'longName': '', 'shortName': '', 'code': ''}, {'types': ['ROAD_NAME'], 'longName': '세종대로', 'shortName': '세종대로', 'code': ''}, {'types': ['BUILDING_NUMBER'], 'longName': '110', 'shortName': '110', 'code': ''}, {'types': ['BUILDING_NAME'], 'longName': '서울특별시청', 'shortName': '서울특별시청', 'code': ''}, {'types': ['LAND_NUMBER'], 'longName': '31', 'shortName': '31', 'code': ''}, {'types': ['POSTAL_CODE'], 'l

In [ ]:
# 처음 실행할 때는 RUN_NAVER_GEOCODING=True, GEOCODE_LIMIT=10으로 소량 테스트한다.
# 테스트 성공 후 GEOCODE_LIMIT=None으로 바꾸면 전체 고유 주소를 처리한다.
# GEOCODE_FORCE=False이면 기존 성공 캐시는 재사용하고, GEOCODE_RETRY_CACHE_STATUSES에 해당하는 실패 주소만 재시도한다.
RUN_NAVER_GEOCODING = True
GEOCODE_LIMIT = None
GEOCODE_FORCE = False
GEOCODE_MAX_WORKERS = 5
GEOCODE_MAX_RETRIES = 3
GEOCODE_RETRY_BASE_DELAY = 1.0
GEOCODE_RETRY_CACHE_STATUSES = ('error',)

if RUN_NAVER_GEOCODING:
    geocode_unique_addresses(
        input_path=apt_path,
        output_path=apt_geocoded_path,
        address_col='full_road_address',
        sleep_seconds=0.1,
        limit=GEOCODE_LIMIT,
        force=GEOCODE_FORCE,
        max_workers=GEOCODE_MAX_WORKERS,
        max_retries=GEOCODE_MAX_RETRIES,
        retry_base_delay=GEOCODE_RETRY_BASE_DELAY,
        retry_cache_statuses=GEOCODE_RETRY_CACHE_STATUSES,
    )
    apt_with_coordinates = merge_coordinates(
        apartment_path=apt_path,
        geocoded_path=apt_geocoded_path,
        output_path=apt_with_coordinates_path,
        address_col='full_road_address',
    )
    display(apt_with_coordinates['geocode_status'].value_counts(dropna=False))
else:
    print('RUN_NAVER_GEOCODING=False: API 호출은 실행하지 않습니다.')


In [ ]:
large_marts[['address']].dropna().drop_duplicates().to_csv(
    large_mart_address_path, index=False, encoding='utf-8-sig'
)

if RUN_NAVER_GEOCODING:
    geocode_unique_addresses(
        input_path=large_mart_address_path,
        output_path=large_mart_geocoded_path,
        address_col='address',
        sleep_seconds=0.1,
        limit=None,
        force=GEOCODE_FORCE,
        max_workers=GEOCODE_MAX_WORKERS,
        max_retries=GEOCODE_MAX_RETRIES,
        retry_base_delay=GEOCODE_RETRY_BASE_DELAY,
        retry_cache_statuses=GEOCODE_RETRY_CACHE_STATUSES,
    )

if large_mart_geocoded_path.exists():
    large_mart_coordinates = pd.read_csv(large_mart_geocoded_path, encoding='utf-8-sig')
    large_marts_for_features = large_marts.merge(
        large_mart_coordinates[['address', 'latitude', 'longitude', 'geocode_status']],
        on='address',
        how='left',
    )
    missing_large_mart_address = large_marts_for_features['address'].isna()
    large_marts_for_features.loc[missing_large_mart_address, 'geocode_status'] = 'missing_address'
    large_marts_for_features.to_csv(large_mart_with_coordinates_path, index=False, encoding='utf-8-sig')
    print(f'대형마트 좌표 보유: {large_marts_for_features["latitude"].notna().sum():,}/{len(large_marts_for_features):,}')
    display(large_marts_for_features['geocode_status'].value_counts(dropna=False))
else:
    large_marts_for_features = None
    print('대형마트 지오코딩 결과가 아직 없습니다.')


## 6. 거리 기반 파생변수 생성

아파트 좌표가 붙은 `seoul_apt_trade_2025_with_coordinates.csv`가 있으면 다음 파생변수를 계산해 `data/processed/seoul_apt_trade_2025_features.csv`로 저장한다.

- `nearest_subway_distance_km`: 아파트 좌표와 전체 역 좌표 사이의 최단 거리
- `distance_to_cbd_km`, `distance_to_ybd_km`, `distance_to_gbd_km`: 3대 업무지구별 거리
- `nearest_business_district_distance_km`: CBD/YBD/GBD 중심 좌표 중 최단 거리
- `nearest_business_district`: 가장 가까운 업무지구 코드
- `hospital_count_within_1km`: 반경 1km 이내 상급종합병원/종합병원 수
- `nearest_hospital_distance_km`: 상급종합병원/종합병원까지의 최단 거리
- `large_mart_count_within_1km`: 반경 1km 이내 대형마트 수

중심업무지구 좌표는 이 노트북에서 정의한 `CBD_CENTERS` 상수에 고정해 사용한다.

좌표가 없는 거래 행은 이 단계에서 바로 삭제하지 않고 `seoul_apt_geocode_unresolved.csv`로 별도 저장한다. 전체 feature 파일에는 원본 행을 유지하고, 모델링에 바로 사용할 수 있는 좌표 완비 행은 `seoul_apt_trade_2025_features_modeling_ready.csv`로 분리해 저장한다.

In [ ]:
pd.DataFrame(CBD_CENTERS).T

,name,description,address,latitude,longitude
CBD,도심권,시청/광화문 일대,서울특별시 중구 세종대로 110,37.5665,126.978
YBD,여의도권,여의도 일대,서울특별시 영등포구 의사당대로 1,37.5259,126.9209
GBD,강남권,강남역 사거리 일대,서울특별시 강남구 강남대로 396,37.4979,127.0276


In [ ]:
stations_for_features = station_master.rename(
    columns={'역사_ID': 'station_id', '역사명': 'station_name', '호선': 'line', '위도': 'latitude', '경도': 'longitude'}
)[['station_id', 'station_name', 'line', 'latitude', 'longitude']]

hospitals_for_features = seoul_hospitals.dropna(subset=['latitude', 'longitude']).copy()

if large_mart_with_coordinates_path.exists():
    large_marts_for_features = pd.read_csv(large_mart_with_coordinates_path, encoding='utf-8-sig')
    large_marts_for_features = large_marts_for_features.dropna(subset=['latitude', 'longitude'])
else:
    large_marts_for_features = None

print(f'역 좌표: {len(stations_for_features):,}')
print(f'병원 좌표: {len(hospitals_for_features):,}')
print('대형마트 좌표:', '없음' if large_marts_for_features is None else f'{len(large_marts_for_features):,}')

역 좌표: 783
병원 좌표: 59
대형마트 좌표: 62


In [ ]:
if apt_with_coordinates_path.exists():
    apt_with_coordinates = pd.read_csv(apt_with_coordinates_path, encoding='utf-8-sig')

    missing_address_mask = apt_with_coordinates['full_road_address'].isna()
    missing_status_mask = apt_with_coordinates['geocode_status'].isna()
    coordinate_missing_mask = apt_with_coordinates[['latitude', 'longitude']].isna().any(axis=1)

    apt_with_coordinates.loc[missing_address_mask & missing_status_mask, 'geocode_status'] = 'missing_address'
    apt_with_coordinates.loc[~missing_address_mask & missing_status_mask & coordinate_missing_mask, 'geocode_status'] = 'not_geocoded'

    coordinate_count = (~coordinate_missing_mask).sum()
    print(f'아파트 좌표 보유: {coordinate_count:,}/{len(apt_with_coordinates):,}')
    display(apt_with_coordinates['geocode_status'].value_counts(dropna=False))

    unresolved_columns = [
        'apartment_name',
        'gu',
        'law_dong',
        'full_road_address',
        'geocode_status',
        'matched_address',
        'error_message',
        'latitude',
        'longitude',
    ]
    available_unresolved_columns = [col for col in unresolved_columns if col in apt_with_coordinates.columns]
    geocode_unresolved = apt_with_coordinates.loc[coordinate_missing_mask, available_unresolved_columns].copy()
    geocode_unresolved.to_csv(geocode_unresolved_path, index=False, encoding='utf-8-sig')
    print(f'좌표 결측 점검 파일 저장: {geocode_unresolved_path} ({len(geocode_unresolved):,}행)')

    if coordinate_count == 0:
        print('좌표가 하나도 없어 거리 기반 파생변수를 생성하지 않습니다.')
        print('Naver 지오코딩 결과의 geocode_status/error_message를 먼저 확인하세요.')
    else:
        apt_features = add_accessibility_features(
            apt_with_coordinates,
            stations=stations_for_features,
            hospitals=hospitals_for_features,
            large_marts=large_marts_for_features,
        )
        apt_features.to_csv(features_output_path, index=False, encoding='utf-8-sig')

        modeling_ready_required_columns = [
            'latitude',
            'longitude',
            'nearest_subway_distance_km',
            'nearest_business_district_distance_km',
            'nearest_hospital_distance_km',
            'large_mart_count_within_1km',
        ]
        available_required_columns = [col for col in modeling_ready_required_columns if col in apt_features.columns]
        apt_features_modeling_ready = apt_features.dropna(subset=available_required_columns).copy()
        apt_features_modeling_ready.to_csv(features_modeling_ready_path, index=False, encoding='utf-8-sig')

        print(f'saved: {features_output_path}')
        print(f'saved: {features_modeling_ready_path}')
        print(f'전체 feature 데이터: {apt_features.shape[0]:,}행 x {apt_features.shape[1]:,}열')
        print(f'모델링용 좌표 완비 데이터: {len(apt_features_modeling_ready):,}행')
        print(f'모델링 전 분리된 좌표/입지 결측 행: {len(apt_features) - len(apt_features_modeling_ready):,}행')

        preview_columns = [
            'apartment_name',
            'full_road_address',
            'latitude',
            'longitude',
            'nearest_subway_distance_km',
            'nearest_business_district',
            'nearest_business_district_distance_km',
            'hospital_count_within_1km',
            'nearest_hospital_distance_km',
        ]
        display(apt_features_modeling_ready[preview_columns].head())
else:
    print(f'아파트 좌표 파일이 아직 없습니다: {apt_with_coordinates_path}')
    print('위 Naver 지오코딩 셀에서 RUN_NAVER_GEOCODING=True로 실행한 뒤 다시 실행하세요.')


## 7. 정제 결과 저장

In [ ]:
hospital_summary.to_csv(INTERIM_DIR / 'hospital_2025_quarter_comparison.csv', index=False, encoding='utf-8-sig')
hospital_diffs.to_csv(INTERIM_DIR / 'hospital_2025_quarter_differences.csv', index=False, encoding='utf-8-sig')
seoul_hospitals.to_csv(INTERIM_DIR / 'seoul_hospitals_tertiary_general_2025_12.csv', index=False, encoding='utf-8-sig')
large_marts.to_csv(INTERIM_DIR / 'seoul_large_marts_cleaned.csv', index=False, encoding='utf-8-sig')

sorted(path.name for path in INTERIM_DIR.glob('*.csv'))

['hospital_2025_quarter_comparison.csv',
 'hospital_2025_quarter_differences.csv',
 'seoul_apt_address_geocoded.csv',
 'seoul_apt_trade_2025_basic_cleaned.csv',
 'seoul_hospitals_tertiary_general_2025_12.csv',
 'seoul_large_mart_addresses.csv',
 'seoul_large_mart_addresses_geocoded.csv',
 'seoul_large_marts_cleaned.csv',
 'seoul_large_marts_geocoded.csv',
 'seoul_subway_stations_cleaned.csv']